In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load AquaSensor data
df = pd.read_csv(
    "../data/raw/aquasensor_all_sensors.csv",
    parse_dates=['timestamp']
)
df = df.sort_values(['sensor_id', 'timestamp'])

print("=== AQUASENSOR DATA ===")
print(f"Total rows:  {len(df)}")
print(f"Sensors:     {df['sensor_id'].unique()}")
print(f"Date range:  {df['timestamp'].min()} "
      f"to {df['timestamp'].max()}")
print(f"Columns:     {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic stats:\n{df.describe().round(2)}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AquaSensor — River Derwent Overview',
             fontsize=14, fontweight='bold')

# Scatter: temperature vs DO
axes[0].scatter(
    df['temperature'],
    df['dissolved_oxygen_mgl'],
    alpha=0.2, s=3, color='blue'
)
axes[0].set_xlabel('Water Temperature (°C)')
axes[0].set_ylabel('Dissolved Oxygen (mg/L)')
axes[0].set_title('Temperature vs DO\n(inverse relationship expected)')
corr = df['temperature'].corr(df['dissolved_oxygen_mgl'])
axes[0].text(0.05, 0.95, f'Correlation: {corr:.3f}',
             transform=axes[0].transAxes,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightblue'))

# Time series for each sensor
colors = ['blue', 'green', 'red']
for i, sensor in enumerate(df['sensor_id'].unique()):
    s = df[df['sensor_id'] == sensor]
    axes[1].plot(
        s['timestamp'], s['dissolved_oxygen_mgl'],
        label=sensor, alpha=0.7,
        linewidth=0.5, color=colors[i]
    )
axes[1].axhline(
    y=4.0, color='red', linestyle='--',
    label='Critical threshold (4 mg/L)'
)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Dissolved Oxygen (mg/L)')
axes[1].set_title('DO Over Time — All Sensors')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../figures/01_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCorrelation (temp vs DO): {corr:.3f}")
print("Expected: negative value between -0.7 and -0.9")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

sensors = df['sensor_id'].unique()
colors  = ['blue', 'green', 'orange']
names   = ['Derwent 13 (sensor022)',
           'Derwent 13-50 (941115)',
           'Derwent 21 (941205)']

for i, sensor in enumerate(sensors):
    s = df[df['sensor_id'] == sensor].copy()

    ax = axes[i]
    ax2 = ax.twinx()

    ax.plot(s['timestamp'], s['dissolved_oxygen_mgl'],
            color=colors[i], linewidth=0.6,
            label='DO (mg/L)', alpha=0.8)
    ax2.plot(s['timestamp'], s['temperature'],
             color='red', linewidth=0.5,
             label='Temperature (°C)', alpha=0.5)

    ax.axhline(y=4.0, color='black',
               linestyle='--', alpha=0.5,
               label='Critical (4 mg/L)')
    ax.set_ylabel('DO (mg/L)', color=colors[i])
    ax2.set_ylabel('Temperature (°C)', color='red')
    ax.set_title(names[i])
    ax.legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('../figures/02_per_sensor.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=== ANOMALY CHECK ===")
print(f"\nReadings below 4 mg/L (critical threshold):")
critical = df[df['dissolved_oxygen_mgl'] < 4.0]
print(critical.groupby('sensor_id').size())

print(f"\nReadings below 6 mg/L (warning threshold):")
warning = df[df['dissolved_oxygen_mgl'] < 6.0]
print(warning.groupby('sensor_id').size())

print(f"\nDO statistics per sensor:")
print(df.groupby('sensor_id')['dissolved_oxygen_mgl'].agg(
    ['mean', 'min', 'max', 'std']
).round(2))

print(f"\nTemperature statistics per sensor:")
print(df.groupby('sensor_id')['temperature'].agg(
    ['mean', 'min', 'max', 'std']
).round(2))

In [ ]:
weather = pd.read_csv(
    "../data/raw/weather_data.csv",
    parse_dates=['timestamp']
)
print("=== WEATHER DATA ===")
print(f"Rows:    {len(weather)}")
print(f"Columns: {weather.columns.tolist()}")
print(f"\nStats:\n{weather.describe().round(2)}")

# Plot sunshine and air temp
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(weather['timestamp'],
             weather['sunshine_wm2'],
             color='orange', linewidth=0.5)
axes[0].set_ylabel('Sunshine (W/m²)')
axes[0].set_title('Sunshine over time')

axes[1].plot(weather['timestamp'],
             weather['air_temp_c'],
             color='red', linewidth=0.5)
axes[1].set_ylabel('Air Temperature (°C)')
axes[1].set_title('Air Temperature over time')

plt.tight_layout()
plt.savefig('../figures/03_weather_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()